In [ ]:
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN
!pip install gfpgan>=1.3.5
!pip install basicsr>=1.3.3.11
!pip install facexlib>=0.2.0.3
!pip install gfpgan>=0.2.1
!pip install -r requirements.txt
!python setup.py develop

Cloning into 'Real-ESRGAN'...
remote: Enumerating objects: 759, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 759 (delta 106), reused 98 (delta 98), pack-reused 638 (from 1)
Receiving objects: 100% (759/759), 5.38 MiB | 7.65 MiB/s, done.
Resolving deltas: 100% (415/415), done.
/content/Real-ESRGAN
/usr/local/lib/python3.12/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
!!

        ********************************************************************************
        Requirements should be satisfied by a PEP 517 installer.
        If you are using pip, you can try `pip install --use-pep517`.
        ********************************************************************************

!!
  dist.fetch_build_eggs(dist.setup_requires)
running develop
/usr/local/lib/python3.12/dist-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarni

In [ ]:
import os
os.makedirs('weights', exist_ok=True)
!wget https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P ./weights
!wget https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth -P ./weights

In [ ]:
import os
import sys

# Traceback'te belirtilen dosya yolu
DEGRADATIONS_FILE = "/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py"

try:
    with open(DEGRADATIONS_FILE, 'r') as f:
        content = f.read()

    # Yama 1: Hatalı import satırını devre dışı bırak
    old_import_line = "from torchvision.transforms.functional_tensor import rgb_to_grayscale"
    new_import_line = "# from torchvision.transforms.functional_tensor import rgb_to_grayscale # YAMA: Colab uyumluluğu"
    content = content.replace(old_import_line, new_import_line)

    # Yama 2: Eğer içerideki kod bu fonksiyonu kullanıyorsa onu da devre dışı bırak (örn. gfpgan kullanıyorsa)
    # Bu adımı sadece, import'u devre dışı bıraktıktan sonra başka bir hata alırsak deneriz.
    # Şu an sadece import'u kapattık. Dosya içeriğini tekrar yazıyoruz.

    with open(DEGRADATIONS_FILE, 'w') as f:
        f.write(content)

    print(f"✅ basicsr import yaması uygulandı: {DEGRADATIONS_FILE}")
    print("⚠️ Hata devam ederse, içerideki fonksiyon çağrısını da düzenlememiz gerekecek.")

except FileNotFoundError:
    print(f"❌ Hata: Dosya yolu bulunamadı. Kurulumdan sonra bu dosyayı tekrar arayın.")
except Exception as e:
    print(f"❌ Dosya düzenlenirken hata oluştu: {e}")

✅ basicsr import yaması uygulandı: /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py
⚠️ Hata devam ederse, içerideki fonksiyon çağrısını da düzenlememiz gerekecek.


In [ ]:
from basicsr.archs.rrdbnet_arch import RRDBNet
import cv2

from realesrgan import RealESRGANer
from realesrgan.archs.srvgg_arch import SRVGGNetCompact
from gfpgan import GFPGANer
import tempfile
import base64
import numpy as np
import io
from PIL import Image

In [ ]:
# GPU ve VRAM Optimizasyonu (A100 ve CPU Uyumlu)
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# GPU varsa FP16 (yarı hassasiyet) aktif edilir, CPU'da hata vermemesi için False yapılır.
use_half = torch.cuda.is_available()

model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
netscale = 4
model_path = os.path.join("weights", "RealESRGAN_x4plus.pth")
upsampler = RealESRGANer(
            scale=netscale,
            model_path=model_path,
            model=model,
            tile=0,
            tile_pad=10,
            pre_pad=0,
            half=use_half)

gfpgan_path = os.path.join("weights", "GFPGANv1.4.pth")
restorer = GFPGANer(
            model_path=gfpgan_path,
            upscale=netscale,
            arch='clean',
            channel_multiplier=2,
            bg_upsampler=upsampler)

In [ ]:
#@title 🐺 Orioninsist | Profesyonel Yapay Zeka Upscaler (300 DPI Entegrasyonlu)
#@markdown Bu araç, model resimlerinizi veya tişört tasarımlarınızı yapay zeka ile 4-8 kat büyüterek 300 DPI baskı kalitesine ulaştırır.
#@markdown Kodları gizlemek için hücre başlığına çift tıklayabilir ve işlemleri doğrudan form üzerinden yürütebilirsiniz.
#@markdown ---

scale = 4 #@param {type:"slider", min:2, max:8, step:1}
#@markdown *   *İpucu:* Görselin kaç kat büyütüleceğini seçin (300 DPI baskı için 4 idealdir).

use_face_restore = True #@param {type:"boolean"}
#@markdown *   ⚠️ **ÖNEMLİ KURAL**: Eğer tişört üzerine basılacak grafikleri/logoları büyütüyorsanız bu ayarı kesinlikle **KAPATIN (False)**. 
#@markdown *   Manken görsellerini (Freya) büyütürken yüz hatlarının pürüzsüzleşmesi için bu ayarı **AÇIN (True)**.

from google.colab import files
import os
import cv2
import numpy as np
from PIL import Image

def process_and_upscale(image_path, output_path, scale, use_face_restore=False):
    img = cv2.imread(image_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        print(f"❌ Hata: {image_path} yüklenemedi.")
        return False

    channels = img.shape[2] if len(img.shape) == 3 else 1
    if channels == 1:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        channels = 3

    has_alpha = (channels == 4)

    try:
        if use_face_restore:
            if has_alpha:
                bgr = img[:, :, :3]
                alpha = img[:, :, 3]

                _, _, restored_bgr = restorer.enhance(
                    bgr,
                    has_aligned=False,
                    only_center_face=False,
                    paste_back=True,
                    weight=0.5
                )

                alpha_bgr = cv2.merge([alpha, alpha, alpha])
                upscaled_alpha_bgr, _ = upsampler.enhance(alpha_bgr, outscale=scale)
                upscaled_alpha = upscaled_alpha_bgr[:, :, 0]

                b, g, r = cv2.split(restored_bgr)
                output = cv2.merge([b, g, r, upscaled_alpha])
            else:
                _, _, output = restorer.enhance(
                    img,
                    has_aligned=False,
                    only_center_face=False,
                    paste_back=True,
                    weight=0.5
                )
        else:
            output, _ = upsampler.enhance(img, outscale=scale)

        # Sonucu kaydet ve 300 DPI meta verisini enjekte et
        if has_alpha:
            pil_img = Image.fromarray(cv2.cvtColor(output, cv2.COLOR_BGRA2RGBA))
            pil_img.save(output_path, dpi=(300, 300))
        else:
            pil_img = Image.fromarray(cv2.cvtColor(output, cv2.COLOR_BGR2RGB))
            pil_img.save(output_path, dpi=(300, 300))
        return True
    except RuntimeError as error:
        print('Hata oluştu:', error)
        return False
    except Exception as e:
        print(f"Beklenmedik hata: {e}")
        return False

print("Lütfen bilgisayarınızdan büyütmek istediğiniz görsel(ler)i seçin:")
uploaded = files.upload()

if len(uploaded) == 0:
    print("❌ Hiçbir dosya seçilmedi.")
else:
    for filename in uploaded.keys():
        print("-" * 60)
        print(f"⏳ İşleniyor: {filename} ...")

        name, ext = os.path.splitext(filename)
        output_filename = f"{name}_upscaled{ext}"

        success = process_and_upscale(filename, output_filename, scale, use_face_restore)

        if success:
            print(f"✅ Başarılı! Bilgisayarınıza indiriliyor: {output_filename}")
            files.download(output_filename)
